In [2]:
import torch
import numpy as np
import gc
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline
from rdflib import Graph, RDFS
from sklearn.metrics.pairwise import cosine_similarity
from fuzzywuzzy import process
import pickle
import os
import shutil

In [ ]:
class AnimeChatbot:
    def __init__(self, model, entity_to_id, anime_data, label_dict):
        self.model = model
        self.entity_to_id = entity_to_id
        self.anime_data = anime_data
        self.id_to_name = label_dict
        
        # Reconstruct valid_ids from the data we saved
        # (We only saved data for valid anime, so the keys are the valid IDs)
        self.valid_ids = set(anime_data.keys())

        # Pre-compute embeddings (Move to CPU for inference)
        self.model.eval()
        with torch.no_grad():
            self.all_embeddings = self.model.entity_representations[0](indices=None).detach().cpu().numpy()

        # Build Search List
        self.name_to_id = {}
        for entity_id in self.entity_to_id.keys():
            if entity_id in self.valid_ids:
                readable_name = self.id_to_name.get(entity_id, entity_id)
                self.name_to_id[readable_name] = entity_id

        self.searchable_names = list(self.name_to_id.keys())
        
        # Manual Aliases
        self.aliases = {
            "demon slayer": "Kimetsu no Yaiba",
            "attack on titan": "Shingeki no Kyojin",
            "my hero academia": "Boku no Hero Academia"
        }

    def recommend(self, user_input):
        clean_input = user_input.lower().strip()
        if clean_input in self.aliases:
            user_input = self.aliases[clean_input]

        # Use token_set_ratio to handle "Demon Slayer" -> "Demon Slayer: Kimetsu no Yaiba"
        best_match, score = process.extractOne(
            user_input, 
            self.searchable_names, 
            scorer=fuzz.token_set_ratio
        )
        
        if score < 60: 
            return f"I couldn't find '{user_input}'. Did you mean: {best_match}?"

        target_id = self.name_to_id[best_match]
        
        # Run Calculation
        recs = self._calculate(target_id)

        response = f"Found: **{best_match}**\nTop recommendations:\n"
        for i, (name, final_score) in enumerate(recs, 1):
            response += f"{i}. **{name}** ({int(final_score*100)}%)\n"
        return response

    def _calculate(self, target, alpha=0.5, top_k=10):
        target_idx = self.entity_to_id[target]
        target_emb = self.all_embeddings[target_idx].reshape(1, -1)
        t_feats = self.anime_data.get(target)
        
        sim_scores = cosine_similarity(target_emb, self.all_embeddings)[0]

        scores = []
        target_readable = self.id_to_name.get(target, target).lower()
        
        for name in self.anime_data:
            if name == target: continue
            if name not in self.entity_to_id: continue

            readable = self.id_to_name.get(name, name)
            readable = readable.replace("_", " ")
            
            # Filters
            if "movie" in readable.lower(): continue 
            if target_readable.split()[0] in readable.lower(): continue

            # Scores
            idx = self.entity_to_id[name]
            s_transr = (sim_scores[idx] + 1) / 2

            feats = self.anime_data[name]
            u_g = len(t_feats["G"].union(feats["G"]))
            s_g = len(t_feats["G"].intersection(feats["G"])) / u_g if u_g > 0 else 0
            s_e = 1.0 if len(t_feats["E"].intersection(feats["E"])) > 0 else 0.0
            
            s_content = (0.7 * s_g) + (0.3 * s_e) 
            final = (alpha * s_content) + ((1 - alpha) * s_transr)
            
            scores.append((readable, final))

        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

In [17]:
def load_system(folder_path):
    print(f"Loading system from '{folder_path}'...")
    
    # Path to the model file
    model_path = os.path.join(folder_path, "model.pt")
    
    # 1. Load Model
    # We set weights_only=False because PyKEEN saves the whole class structure, 
    # not just the weights. Since you built this model, it is safe.
    model = torch.load(
        model_path, 
        map_location=torch.device('cpu'), 
        weights_only=False 
    )
    
    # 2. Load Triples Factory (For ID mappings)
    with open(os.path.join(folder_path, "triples_factory.pkl"), "rb") as f:
        triples_factory = pickle.load(f)
        
    # 3. Load Data Package (Labels and Features)
    with open(os.path.join(folder_path, "data_package.pkl"), "rb") as f:
        data_pkg = pickle.load(f)
    
    print("System loaded successfully.")
    
    # Initialize Chatbot
    return AnimeChatbot(
        model=model,
        entity_to_id=triples_factory.entity_to_id,
        anime_data=data_pkg["anime_data"],
        label_dict=data_pkg["id_to_label"]
    )

In [18]:
folder_name = "anime_recommender_final" 
    
try:
    bot = load_system(folder_name)
    
    print("\n--- Ready to Chat ---")
    while True:
        user_input = input("\nEnter anime name (or 'q' to quit): ")
        if user_input.lower() == 'q': break
        print(bot.recommend(user_input))
        
except FileNotFoundError:
    print(f"Error: Could not find folder '{folder_name}'. Make sure you extracted the zip file there.")

Loading system from 'anime_recommender_final'...
System loaded successfully.

--- Ready to Chat ---
Found: **THE ONE PIECE**
Top recommendations:
1. **Shaman King** (67%)
2. **Soar High! Isami** (65%)
3. **One Piece** (64%)
4. **Astro Boy** (63%)
5. **Miami Guns** (63%)
6. **Fairy Tail** (62%)
7. **Code:Breaker** (62%)
8. **Hunter × Hunter** (61%)
9. **Beelzebub** (61%)
10. **Sorcerous Stabber Orphen** (60%)

Found: **Demon Slayer: Kimetsu no Yaiba**
Top recommendations:
1. **Tougen Anki** (66%)
2. **MapleStory** (66%)
3. **Seven Deadly Sins** (65%)
4. **The Banished Court Magician Aims to Become the Strongest** (65%)
5. **Drifters** (64%)
6. **Flame of Recca** (63%)
7. **The Witch and the Beast** (63%)
8. **Gachiakuta** (63%)
9. **Taboo Tattoo** (63%)
10. **Naruto: Shippūden** (62%)

Found: **Naruto**
Top recommendations:
1. **One_Piece** (81%)
2. **DGrayman** (81%)
3. **The_Twelve_Kingdoms** (80%)
4. **Claymore** (79%)
5. **Soul_Hunter** (79%)
6. **Hunter_x_Hunter** (79%)
7. **Bakegy